Position sizing comparison: regime x VRP rank (the current scheme in `04_position_sizing.ipynb`) vs. a rolling half-Kelly fraction sized off the strategy's own historical edge. Both run through the same vol-targeting overlay from `05_backtest.ipynb`, so the comparison isolates the sizing signal itself rather than differences in leverage policy.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
from src.vrp_strategy.backtest import raw_strategy_returns, vol_target_scale
from src.vrp_strategy.metrics import performance_metrics, print_metrics, safe_corr

DATA_PROCESSED = ROOT / "data" / "processed"
data = pd.read_csv(DATA_PROCESSED / "positions.csv", index_col=0, parse_dates=True)

TARGET_VOL, VOL_WINDOW, SCALAR_CAP = 0.10, 126, 100.0
KELLY_WINDOW, KELLY_FRACTION = 252, 0.5

# Unit-exposure return: what position = 1.0 would have earned each day
data["unit_ret"] = data["iv_daily"].shift(1) - data["rv_daily"]

# Kelly fraction f* = mean(edge) / variance(edge) on a trailing window,
# shifted so day t only uses info through t-1. Half-Kelly (0.5x) as a
# standard safety margin against estimation error, clipped to [0, 1] to
# stay on the same footing as the existing 0-1 position signal.
roll_mean = data["unit_ret"].rolling(KELLY_WINDOW).mean().shift(1)
roll_var  = data["unit_ret"].rolling(KELLY_WINDOW).var().shift(1)
data["kelly_position"] = (KELLY_FRACTION * roll_mean / roll_var).clip(lower=0, upper=1)

print("Kelly position, full history:")
print(data["kelly_position"].describe())

Kelly position, full history:
count    4460.000000
mean        0.877705
std         0.327457
min         0.000000
25%         1.000000
50%         1.000000
75%         1.000000
max         1.000000
Name: kelly_position, dtype: float64


In [2]:
SPLIT = "2016-01-01"
data = data.dropna(subset=["kelly_position", "position"])
data = data[data.index >= SPLIT]
print(f"Out-of-sample period: {data.index[0].date()} → {data.index[-1].date()}  ({len(data)} rows)")

Out-of-sample period: 2016-01-04 → 2026-05-22  (2612 rows)


In [3]:
variants = {
    "Current (regime x VRP rank)": data["position"],
    "Kelly fraction (half-Kelly)": data["kelly_position"],
}

rows, curves = [], {}
for name, pos in variants.items():
    raw_ret = raw_strategy_returns(pos, data["iv_daily"], data["rv_daily"])
    scaled_ret, _ = vol_target_scale(raw_ret, TARGET_VOL, VOL_WINDOW, SCALAR_CAP)
    scaled_ret = scaled_ret.dropna()
    m = performance_metrics(scaled_ret)
    m["variant"] = name
    rows.append(m)
    curves[name] = scaled_ret
    print_metrics(m, name)

comparison = pd.DataFrame(rows).set_index("variant")[
    ["ann_return", "ann_vol", "sharpe", "sortino", "calmar", "max_drawdown", "win_rate"]
]


Current (regime x VRP rank)
  Annualised return:  31.94%
  Annualised vol:     10.37%
  Sharpe ratio:       3.080
  Sortino ratio:      1.479
  Calmar ratio:       1.627
  Max drawdown:       -19.64%
  Win rate:           76.1%

Kelly fraction (half-Kelly)
  Annualised return:  45.28%
  Annualised vol:     14.71%
  Sharpe ratio:       3.079
  Sortino ratio:      1.615
  Calmar ratio:       1.318
  Max drawdown:       -34.36%
  Win rate:           72.2%


In [4]:
corr = safe_corr(data["position"], data["kelly_position"])
print(f"Correlation between the two signals: {corr:.3f}")

comparison.to_csv(DATA_PROCESSED / "position_sizing_comparison.csv")
print("\nSaved position_sizing_comparison.csv")
print(comparison)

Correlation between the two signals: 0.327

Saved position_sizing_comparison.csv
                             ann_return   ann_vol    sharpe   sortino  \
variant                                                                 
Current (regime x VRP rank)     0.31939  0.103693  3.080135  1.478847   
Kelly fraction (half-Kelly)     0.45283  0.147072  3.078962  1.615276   

                               calmar  max_drawdown  win_rate  
variant                                                        
Current (regime x VRP rank)  1.626543     -0.196361  0.760966  
Kelly fraction (half-Kelly)  1.317812     -0.343622  0.722334  


Kelly sizing estimates its own edge from a trailing year of realised strategy returns, so it reacts slowly. A regime shift only shows up in the Kelly fraction once it's dragged a full year of rolling mean and variance with it, while the regime multiplier reacts within days of the HMM reclassifying the state. That shows up directly in the numbers: Kelly runs bigger and rides further into stress before backing off, which lifts annualised return but also more than doubles the drawdown.

In [5]:
fig, axes = plt.subplots(3, 1, figsize=(14, 11), sharex=True)
fig.suptitle("Position Sizing — Current vs Kelly Fraction", fontsize=14, fontweight="bold")

colors = {"Current (regime x VRP rank)": "#2563eb", "Kelly fraction (half-Kelly)": "#f59e0b"}

for name, ret in curves.items():
    cum = (1 + ret).cumprod()
    axes[0].plot(cum.index, cum.values, color=colors[name], linewidth=1.2, label=name)
axes[0].set_ylabel("Cumulative return")
axes[0].set_title("Cumulative performance")
axes[0].legend(loc="upper left", fontsize=9)

for name, ret in curves.items():
    cum = (1 + ret).cumprod()
    dd = cum / cum.cummax() - 1
    axes[1].plot(dd.index, dd.values * 100, color=colors[name], linewidth=1.0, label=name)
axes[1].set_ylabel("Drawdown (%)")
axes[1].set_title("Drawdown")
axes[1].legend(loc="lower left", fontsize=9)

axes[2].plot(data.index, data["position"], color=colors["Current (regime x VRP rank)"],
             linewidth=0.8, label="Current", alpha=0.8)
axes[2].plot(data.index, data["kelly_position"], color=colors["Kelly fraction (half-Kelly)"],
             linewidth=0.8, label="Kelly", alpha=0.8)
axes[2].set_ylabel("Position size")
axes[2].set_title("Raw position signal (pre vol-targeting)")
axes[2].legend(loc="upper left", fontsize=9)

for ax in axes:
    ax.xaxis.set_major_locator(mdates.YearLocator(2))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.grid(axis="x", linestyle="--", linewidth=0.4, alpha=0.5)

plt.tight_layout()
plt.savefig(DATA_PROCESSED / "position_sizing_comparison.png", dpi=150, bbox_inches="tight")
plt.show()